# SumExplicitStack (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_cpp.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `SumExplicitStack`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [1]:
%%writefile explicit_stack.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int maxP = 0;   // massima dimensione della pila P

typedef pair<int, int> Pair;

// SumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita
long long SumExplicitStack(int *a, int n)
{
    long long s = 0;
    vector<Pair> P;                 // pila dei sotto-array da sommare
    P.push_back({0, n - 1});
    maxP = max(maxP, (int)P.size());
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];           // caso base
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            maxP = max(maxP, (int)P.size());
        }
    }
    return s;
}

string FmtP(const vector<Pair> &P)
{
    string s = "[";
    for (size_t i = 0; i < P.size(); i++)
    {
        if (i > 0) s += ",";
        s += "(" + to_string(P[i].first) + "," + to_string(P[i].second) + ")";
    }
    return s + "]";
}

long long SumExplicitStackTrace(int *a, int n)
{
    long long s = 0;
    vector<Pair> P = {{0, n - 1}};
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];
            printf("pop (%d,%d): caso base, s=%lld => P=%s\n", l, r, s, FmtP(P).c_str());
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            printf("pop (%d,%d): mid=%d, push (%d,%d) e (%d,%d) => P=%s, s=%lld\n", l, r, mid, mid + 1, r, l, mid, FmtP(P).c_str(), s);
        }
    }
    return s;
}

// ---- le altre somme del codice del corso, con la profondità ----
int depth = 0;
int maxDepth = 0;

long long sumIt(int *a, int n)
{
    long long s = 0;
    for (int i = 0; i < n; i++)
    {
        s = s + a[i];
    }
    return s;
}

long long sumRecHalving(int *a, int l, int r)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long sumRecDecrease(int *a, int n)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (n == 0)
    {
        result = 0;
    }
    else
    {
        result = sumRecDecrease(a, n - 1) + a[n - 1];
    }
    depth--;
    return result;
}

// ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
int stepNo = 0;
int maxActive = 0;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

void Emit(const vector<string> &stack, const string &action)
{
    stepNo++;
    if ((int)stack.size() > maxActive) maxActive = stack.size();
    string st;
    for (size_t i = 0; i < stack.size(); i++)
    {
        if (i > 0) st += " ";
        st += stack[i];
    }
    printf("%2d | %s | %s\n", stepNo, st.c_str(), action.c_str());
}

long long SumRecDecreaseTrace(int *a, int n, vector<string> &stack)
{
    stack.push_back("s(" + to_string(n) + ")");
    if (n == 0)
    {
        Emit(stack, "n=0: return 0");
        stack.pop_back();
        return 0;
    }
    Emit(stack, "n=" + to_string(n) + " != 0: chiama s(" + to_string(n - 1) + "), poi somma a[" + to_string(n - 1) + "]");
    long long r = SumRecDecreaseTrace(a, n - 1, stack);
    Emit(stack, "return " + to_string(r) + "+a[" + to_string(n - 1) + "]=" + to_string(r + a[n - 1]));
    stack.pop_back();
    return r + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {4, 7, 1, 9};
    int n = 4;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("SumExplicitStack(a, 4):\n");
    long long res = SumExplicitStackTrace(a, n);
    maxP = 0;
    SumExplicitStack(a, n);
    printf("risultato %lld, dimensione massima di P = %d\n", res, maxP);
    printf("\n");
    printf("sumIt                = %lld\n", sumIt(a, n));
    maxDepth = 0;
    long long r1 = sumRecHalving(a, 0, n - 1);
    printf("sumRecHalving        = %lld   profondità massima = %d\n", r1, maxDepth);
    maxDepth = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease       = %lld   profondità massima = %d\n", r2, maxDepth);

    int b[] = {4, 7, 1};
    printf("\n");
    printf("a = %s   sumRecDecrease(a, 3)\n", Fmt(b, 3).c_str());
    printf(" # | pila delle chiamate attive | azione\n");
    vector<string> stack;
    res = SumRecDecreaseTrace(b, 3, stack);
    printf("risultato %lld, chiamate attive al massimo = %d = n+1\n", res, maxActive);
    return 0;
}
#endif

Writing explicit_stack.cpp


In [2]:
!g++ -std=c++17 -O2 -o explicit_stack explicit_stack.cpp && ./explicit_stack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [3]:
%%writefile prova_explicit_stack.cpp
#define NOMAIN
#include "explicit_stack.cpp"

int main()
{
    // Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
    vector<int> c(100);
    for (int i = 0; i < 100; i++) c[i] = i + 1;
    maxP = 0;
    long long s = SumExplicitStack(c.data(), 100);
    printf("%lld %d\n", s, maxP);
    return 0;
}

Writing prova_explicit_stack.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_explicit_stack prova_explicit_stack.cpp && ./prova_explicit_stack

5050 8


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).